## Etapa 2
Com base na última etapa, o modelo definido como base para fine-tuning foi o `convnext_tiny`

O transform tem como objetivo manipular as imagens que serão introduzidas no modelo para treino, o objetivo é testar diferentes combinações e recursos presentes no transform para descobrir o que melhor se encaixa para a solução.

In [10]:
import numpy as np

import torch
import torch.nn as nn

from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import f1_score, balanced_accuracy_score

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


### Data

In [11]:
from dotenv import load_dotenv
import os

In [18]:
if load_dotenv():
    print("Env carregado com sucesso.") 

    DATA_DIR = os.getenv("DATA_PATH")
    BATCH_SIZE = 32

    print(DATA_DIR)
else:
    raise ImportError("Erro ao carregar env")

Env carregado com sucesso.
/media/kaua-matheus/HD320/Data/Agroscope/Especialista/Corn


### Transform

In [13]:
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(.85, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225])
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

In [15]:
full_ds = ImageFolder(DATA_DIR)
num_classes = len(full_ds)

In [16]:
n = len(full_ds)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

In [17]:
train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

In [19]:
val_ds.dataset.transform = eval_tf
test_ds.dataset.transform = eval_tf

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [20]:
print("Classes:", full_ds.classes)
print(f"Split -> train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

Classes: ['Healthy', 'Rust_Blight', 'Rust_Common']
Split -> train=2051 val=439 test=441


### Modelo